# Nutrify — Fine-tune Llama 3.2 1B para estimación de calorías
**Ejecutar TODAS las celdas en orden. Asegurate de tener Runtime → T4 GPU.**

In [ ]:
%%capture
# Celda 1: Instalar dependencias (~2 min)
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
# Celda 2: Cargar modelo Llama 3.2 1B + LoRA
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Modelo cargado con LoRA!")

In [ ]:
# Celda 3: Subir dataset
from google.colab import files
import json

uploaded = files.upload()  # Subí dataset.jsonl

dataset_lines = []
with open("dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            dataset_lines.append(json.loads(line))

print(f"Cargadas {len(dataset_lines)} entradas de entrenamiento")
print(f"Ejemplo: {dataset_lines[0]}")

In [ ]:
# Celda 4: Formatear dataset
from datasets import Dataset

alpaca_prompt = """### Instruction:
{}

### Input:
{}

### Response:
{}"""

def format_prompts(examples):
    texts = []
    for instruction, input_text, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        text = alpaca_prompt.format(instruction, input_text, output) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

dataset = Dataset.from_list(dataset_lines)
dataset = dataset.map(format_prompts, batched=True)

print(f"Dataset listo: {len(dataset)} ejemplos")

In [ ]:
# Celda 5: TRAINING (~20-30 min)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(f"\nTraining completo!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Tiempo: {trainer_stats.metrics['train_runtime']:.0f}s")

In [ ]:
# Celda 6: Test del modelo
FastLanguageModel.for_inference(model)

test_foods = [
    "alfajor rasta",
    "dos milanesas con puré",
    "café con leche y tres medialunas",
    "big mac con papas medianas",
    "yogur ser firme",
    "fernet con coca",
    "tres empanadas de carne",
]

for food in test_foods:
    prompt = alpaca_prompt.format("Estimá las calorías de esta comida", food, "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=128,
            do_sample=False,
            use_cache=False,
        )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = result.split("### Response:\n")[-1].strip()
    print(f"{food:40s} → {response}")

In [ ]:
# Celda 7: Guardar modelo mergeado
model.save_pretrained_merged(
    "nutrify-merged",
    tokenizer,
    save_method="merged_16bit",
)
print("Modelo mergeado guardado!")

In [ ]:
# Celda 8: Preparar conversor GGUF
# Clonamos llama.cpp e instalamos SU versión de gguf (misma versión = sin incompatibilidades)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
!pip install -q ./llama.cpp/gguf-py
print("Conversor listo!")

In [ ]:
# Celda 9: Convertir a GGUF Q8 directo (sin compilar C++)
!python llama.cpp/convert_hf_to_gguf.py nutrify-merged --outfile nutrify-q8.gguf --outtype q8_0

import os
size = os.path.getsize("nutrify-q8.gguf") / 1024 / 1024
print(f"nutrify-q8.gguf — {size:.1f} MB")

In [ ]:
# Celda 10: Descargar modelo
from google.colab import files
files.download("nutrify-q8.gguf")